In [ ]:
import kagglehub
import os
import torch
from tqdm import tqdm
from torchvision import models, Module
from torchvision import transforms
from torch.utils.data import DataLoader,Dataset, random_split
from PIL import Image
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt
import random
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import InterpolationMode
import numpy as np

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
!ls /kaggle/input/q3-stage3-2026/dataset/images/

In [ ]:
# Load an image
from PIL import Image
import os
image_path = '/kaggle/input/q3-stage3-2026/dataset/masks/d_r_153_.png'
image = Image.open(image_path)

# Display the image
image

# Define transformation
quick_transform = transforms.Compose([
    transforms.ToTensor()
])

image_tensor = quick_transform(image)

# Show tensor shape
print(image_tensor.shape)  # (Channels, Height, Width)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import glob
from torch.utils.data import Dataset
from PIL import Image

class SUIMDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, mask_transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. Load Image & Mask
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L") # Keep as L (grayscale)

        # 2. Apply Transform
        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        # 3. Remap Mask (we need this because CrossEntropy requires the labels to be consecutive)
        mask = remap_mask(mask)

        return image, mask



from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import DataLoader

# Let's assume the data is not splitted here, we will use only the training folder, then split it.
# 1. Define Paths
root_dir = os.path.join(path, "dataset")
all_images = sorted(glob.glob(f"{root_dir}/images/*.jpg"))
all_masks  = sorted(glob.glob(f"{root_dir}/masks/*.png")) # Check extension!

# 2. Split Data
train_imgs, test_imgs, train_masks, test_masks = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42
)

# need to subtract 1 from the mask to bring it to 0 indexed ranges (CrossEntropyLoss requires labels to start from 0)
class SubtractOne(nn.Module):
  def forward(self, img):
    return img-1

# 3. Define Transform
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()         # ToTensor does two things: Convert to tensor + scaling (divide by 255)
])
transform_mask = transforms.Compose([
    transforms.Resize((256, 256)),
    #SubtractOne(),
    transforms.PILToTensor()       # PILToTensor does one thing: Convert to tensor only (mask should not be scaled!!)
])

# 4. Create Datasets
train_dataset = SUIMDataset(train_imgs, train_masks, transform=transform, mask_transform=transform_mask)
test_dataset  = SUIMDataset(test_imgs,  test_masks,  transform=transform, mask_transform=transform_mask)

# 5. Check Output
img, mask = train_dataset[0]
print(f"Img Shape: {img.shape}")   # [3, 256, 256]
print(f"Mask Shape: {mask.shape}") # [1, 256, 256]
print(f"Unique Classes: {torch.unique(mask)}")


#Create loaders
train_loader = DataLoader(train_dataset, 32, True)
test_loader = DataLoader(test_dataset, 32, False)

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Multi-class segmentation(8 output channel)
).to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]

            outputs = model(images)  # Now [N, H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
#Run

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
### **🔹 Plot Training Loss Curve**
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
# TO DO
import random
import matplotlib.pyplot as plt

model.eval()
# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass

    pred_mask = torch.softmax(pred_mask, dim=1)  # Convert logits to probabilities
    pred_mask = pred_mask.argmax(dim=1).cpu().squeeze().numpy()  # Get class with highest probability

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")  # Show class map
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
